# Modul 01: Data Preprocessing & Benchmark Construction
**Proyek:** Multi-Stage Conversational Passage Retrieval untuk Domain Informasi Obat (Pharma/Medical QA)  
**Berdasarkan Metodologi:** Lin et al. (ACM TOIS 2021) — *Multi-Stage Conversational Passage Retrieval: An Approach to Fusing Term Importance Estimation and Neural Query Rewriting*  

---

### Tujuan Modul Ini:
1. **Inspeksi & Ekstraksi Data Mentah openFDA:** Membaca data partisi `drug-label-0001-of-0014.json` yang memuat label resmi obat dari FDA.
2. **Filtering Klinis Berkualitas Tinggi:** Menyaring obat resep (*Prescription*) dan obat OTC berstandar klinis yang memiliki seksi medis esensial lengkap (*indications, dosage, warnings, adverse reactions, contraindications, drug interactions, overdosage*).
3. **Semantic Passage Chunking & Metadata Tagging:** Memecah seksi medis panjang menjadi pasase ringkas (100–200 kata) dengan injeksi metadata obat `[Drug: ... | Generic: ... | Section: ...]` agar optimal untuk pencarian BM25 dan neural re-ranking.
4. **Penyimpanan Korpus Pasase:** Menyimpan ~5.000 pasase bersih ke `data/processed/passages.jsonl`.
5. **Konstruksi Benchmark Dialog Conversational QA:** Membangun 12 sesi percakapan multi-turn (total 49 turns) yang merepresentasikan fenomena *coreference* dan *omission* beserta label ground-truth `relevant_passage_ids` untuk evaluasi kuantitatif di Modul 04.

## 1. Konfigurasi Lingkungan & Import Library

In [1]:
import os
import json
import re
from collections import Counter
import pandas as pd

# Path direktori data
BASE_DIR = os.path.abspath("..")
RAW_DATA_PATH = os.path.join(BASE_DIR, "data", "raw", "drug-label-0001-of-0014.json", "drug-label-0001-of-0014.json")
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")
PASSAGES_FILE = os.path.join(PROCESSED_DIR, "passages.jsonl")
DIALOGUES_FILE = os.path.join(PROCESSED_DIR, "conversational_dialogues.json")

os.makedirs(PROCESSED_DIR, exist_ok=True)
print(f"Raw data path: {RAW_DATA_PATH} (Exists: {os.path.exists(RAW_DATA_PATH)})")
print(f"Output directory: {PROCESSED_DIR}")

Raw data path: a:\Jurnal IR\Project\data\raw\drug-label-0001-of-0014.json\drug-label-0001-of-0014.json (Exists: True)
Output directory: a:\Jurnal IR\Project\data\processed


## 2. Definisi Seksi Klinis & Fungsi Chunking Pasase

Dokumen FDA label memuat teks medis naratif yang sangat panjang (misal seksi dosis bisa mencapai ribuan kata).  
Untuk mendukung **BM25** dan **Cross-Encoder BERT/MiniLM (maksimal 512 token)**, kita memecah teks menjadi potongan pasase 100–200 kata.
Setiap pasase diinjeksi dengan metadata obat di awal teks: `[Drug: {brand} | Generic: {generic} | Section: {section}]`.

In [2]:
CLINICAL_FIELDS = {
    "indications_and_usage": "INDICATIONS & USAGE",
    "dosage_and_administration": "DOSAGE & ADMINISTRATION",
    "warnings_and_cautions": "WARNINGS & PRECAUTIONS",
    "warnings": "WARNINGS",
    "boxed_warning": "BOXED WARNING",
    "adverse_reactions": "ADVERSE REACTIONS",
    "contraindications": "CONTRAINDICATIONS",
    "drug_interactions": "DRUG INTERACTIONS",
    "overdosage": "OVERDOSAGE",
    "clinical_pharmacology": "CLINICAL PHARMACOLOGY"
}

def clean_text(text: str) -> str:
    """Bersihkan spasi ganda, tag abnormal, dan newline."""
    text = re.sub(r'\s+', ' ', text)
    text = text.replace('\xa0', ' ').strip()
    return text

def chunk_section_text(text: str, max_words: int = 140, min_words: int = 25) -> list:
    """Bagi teks panjang menjadi potongan pasase koheren berukuran 100-200 kata."""
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks = []
    current_chunk = []
    current_len = 0
    
    for s in sentences:
        s = s.strip()
        if not s:
            continue
        words = s.split()
        if current_len + len(words) <= max_words:
            current_chunk.append(s)
            current_len += len(words)
        else:
            if current_chunk and current_len >= min_words:
                chunks.append(" ".join(current_chunk))
            current_chunk = [s]
            current_len = len(words)
            
    if current_chunk and current_len >= min_words:
        chunks.append(" ".join(current_chunk))
    elif current_chunk and chunks:
        chunks[-1] += " " + " ".join(current_chunk)
    elif current_chunk:
        chunks.append(" ".join(current_chunk))
        
    return chunks

## 3. Ekstraksi Korpus Pasase dari openFDA (Target: 5.000 Pasase)

In [3]:
def parse_openfda_dataset(raw_path: str, output_path: str, target_count: int = 5000):
    decoder = json.JSONDecoder()
    passages = []
    processed_drugs = []
    
    with open(raw_path, 'r', encoding='utf-8') as f:
        buffer = ""
        chunk_size = 65536
        while True:
            c = f.read(chunk_size)
            if not c:
                break
            buffer += c
            idx = buffer.find('"results": [')
            if idx != -1:
                buffer = buffer[idx + len('"results": ['):]
                break
                
        record_idx = 0
        p_idx = 1
        
        while len(passages) < target_count:
            buffer = buffer.lstrip(' ,\n\r\t')
            if not buffer or buffer.startswith(']'):
                c = f.read(chunk_size * 4)
                if not c:
                    break
                buffer += c
                buffer = buffer.lstrip(' ,\n\r\t')
                if buffer.startswith(']'):
                    break
            
            try:
                rec, end = decoder.raw_decode(buffer)
                buffer = buffer[end:]
                record_idx += 1
            except Exception:
                c = f.read(chunk_size * 8)
                if not c:
                    break
                buffer += c
                continue
                
            openfda = rec.get("openfda", {})
            brand_names = openfda.get("brand_name", [])
            generic_names = openfda.get("generic_name", [])
            product_type = openfda.get("product_type", [""])[0] if openfda.get("product_type") else ""
            
            if not brand_names and not generic_names:
                continue
                
            brand_str = brand_names[0] if brand_names else generic_names[0]
            generic_str = generic_names[0] if generic_names else brand_str
            
            # Verifikasi seksi klinis
            present_clinical = {}
            for field, section_label in CLINICAL_FIELDS.items():
                if field in rec and rec[field]:
                    val = rec[field]
                    full_sec_txt = " ".join(val) if isinstance(val, list) else str(val)
                    cleaned = clean_text(full_sec_txt)
                    if len(cleaned.split()) >= 25:
                        present_clinical[section_label] = cleaned
                        
            if len(present_clinical) < 2:
                continue
                
            drug_entry = {
                "record_idx": record_idx,
                "brand_name": brand_str,
                "generic_name": generic_str,
                "product_type": product_type,
                "sections": list(present_clinical.keys())
            }
            
            for sec_label, sec_text in present_clinical.items():
                chunks = chunk_section_text(sec_text, max_words=140, min_words=25)
                for c_i, chunk in enumerate(chunks):
                    p_id = f"PASSAGE_{p_idx:05d}"
                    formatted_text = f"[Drug: {brand_str} | Generic: {generic_str} | Section: {sec_label}] {chunk}"
                    
                    p_obj = {
                        "passage_id": p_id,
                        "record_idx": record_idx,
                        "brand_name": brand_str,
                        "generic_name": generic_str,
                        "section": sec_label,
                        "passage_text": formatted_text,
                        "word_count": len(formatted_text.split()),
                        "char_count": len(formatted_text)
                    }
                    passages.append(p_obj)
                    p_idx += 1
                    
                    if len(passages) >= target_count:
                        break
                if len(passages) >= target_count:
                    break
                    
            processed_drugs.append(drug_entry)
            
    with open(output_path, 'w', encoding='utf-8') as f:
        for p in passages:
            f.write(json.dumps(p, ensure_ascii=False) + '\n')
            
    print(f"[DONE] Berhasil memproses {len(processed_drugs)} entitas obat.")
    print(f"[DONE] Berhasil mengekstrak {len(passages)} pasase bersih ke: {output_path}")
    return passages, processed_drugs

# Jalankan ekstraksi
passages, processed_drugs = parse_openfda_dataset(RAW_DATA_PATH, PASSAGES_FILE, target_count=5000)

[DONE] Berhasil memproses 152 entitas obat.
[DONE] Berhasil mengekstrak 5000 pasase bersih ke: a:\Jurnal IR\Project\data\processed\passages.jsonl


## 4. Analisis Statistik Korpus Pasase

In [4]:
df_passages = pd.DataFrame(passages)
print("Ringkasan Statistik Pasase:")
print(df_passages[['word_count', 'char_count']].describe())

print("\nDistribusi Pasase per Seksi Klinis:")
print(df_passages['section'].value_counts())

print("\nContoh 1 Sampel Pasase Terformat:")
sample = df_passages.iloc[0]
print(f"ID: {sample['passage_id']}")
print(f"Drug: {sample['brand_name']}")
print(f"Section: {sample['section']}")
print(f"Text:\n{sample['passage_text']}")

Ringkasan Statistik Pasase:
        word_count  char_count
count  5000.000000  5000.00000
mean    130.742200   923.30660
std      34.346681   247.33878
min      34.000000   212.00000
25%     122.000000   833.00000
50%     137.000000   953.00000
75%     147.000000  1051.00000
max     608.000000  3734.00000

Distribusi Pasase per Seksi Klinis:
section
WARNINGS & PRECAUTIONS     1037
CLINICAL PHARMACOLOGY      1023
ADVERSE REACTIONS           853
DOSAGE & ADMINISTRATION     707
DRUG INTERACTIONS           416
WARNINGS                    358
INDICATIONS & USAGE         220
OVERDOSAGE                  163
BOXED WARNING               117
CONTRAINDICATIONS           106

Contoh 1 Sampel Pasase Terformat:
ID: PASSAGE_00001
Drug: SHISEIDO SYNCHRO SKIN LASTING Liquid FOUNDATION Neutral 5
Section: INDICATIONS & USAGE
Text:
[Drug: SHISEIDO SYNCHRO SKIN LASTING Liquid FOUNDATION Neutral 5 | Generic: OCTINOXATE AND TITANIUM DIOXIDE | Section: INDICATIONS & USAGE] Uses helps prevent sunburn if used a

## 5. Konstruksi Benchmark Dialog Conversational QA

Untuk mengevaluasi modul CQR (**HQE** dan **NTR**) serta perbandingannya dengan kueri manual manusia sesuai metodologi paper TOIS 2021, kita membangun dataset evaluasi percakapan multi-turn:  
* **12 Sesi Percakapan Medis** (Total 49 turns) yang berfokus pada obat-obatan utama dalam korpus (Deferasirox, Celecoxib, Pregabalin, Truvada, Quetiapine, Hydrocodone, Strattera, dll).
* **Fenomena Conversational Search:**
  1. `explicit`: Turn pertama yang menyebutkan entitas obat secara jelas.
  2. `omission`: Turn lanjutan yang menghilangkan nama obat (misal: *"What is the starting dose?"*).
  3. `coreference`: Turn lanjutan yang menggunakan kata ganti / anafora (misal: *"Does it interact with antacids?"*).
* **Ground-Truth Label:** Setiap turn dipetakan ke daftar `relevant_passage_ids` yang memuat jawaban klinis yang tepat untuk menghitung **Recall@K**, **MAP**, dan **NDCG@3**.

In [5]:
with open(DIALOGUES_FILE, 'r', encoding='utf-8') as f:
    benchmark_dialogues = json.load(f)

total_turns = sum(len(s['turns']) for s in benchmark_dialogues)
phenomena = Counter(t['phenomenon'] for s in benchmark_dialogues for t in s['turns'])

print(f"Total Sesi Percakapan: {len(benchmark_dialogues)}")
print(f"Total Giliran (Turns): {total_turns}")
print(f"Distribusi Fenomena Percakapan: {dict(phenomena)}")

print("\nContoh Sesi 1 (Deferasirox):")
s1 = benchmark_dialogues[0]
for t in s1['turns']:
    print(f"Turn {t['turn_id']} [{t['phenomenon'].upper()}]: '{t['raw_utterance']}' -> GT Passages: {t['relevant_passage_ids']}")

Total Sesi Percakapan: 12
Total Giliran (Turns): 49
Distribusi Fenomena Percakapan: {'explicit': 12, 'omission': 15, 'coreference': 22}

Contoh Sesi 1 (Deferasirox):
Turn 1 [EXPLICIT]: 'What is Deferasirox prescribed for?' -> GT Passages: ['PASSAGE_00004', 'PASSAGE_00005', 'PASSAGE_04653']
Turn 2 [OMISSION]: 'What is the recommended starting dose?' -> GT Passages: ['PASSAGE_00006', 'PASSAGE_00008', 'PASSAGE_00013']
Turn 3 [COREFERENCE]: 'What severe warnings are associated with it?' -> GT Passages: ['PASSAGE_00039', 'PASSAGE_00041', 'PASSAGE_04687']
Turn 4 [COREFERENCE]: 'Does this medication interact with antacids?' -> GT Passages: ['PASSAGE_00064', 'PASSAGE_04712', 'PASSAGE_00065']
Turn 5 [OMISSION]: 'What happens if someone accidentally overdoses?' -> GT Passages: ['PASSAGE_00068', 'PASSAGE_04716', 'PASSAGE_00069']


## 6. Kesimpulan & Output Modul 01

Tahap prapemrosesan data telah selesai dengan output terverifikasi:
1. **`data/processed/passages.jsonl`**: 5.000 pasase obat terstruktur dengan metadata injection yang siap diindeks ke dalam **BM25** pada Modul 02.
2. **`data/processed/conversational_dialogues.json`**: 12 sesi percakapan multi-turn dengan ground-truth relevance untuk evaluasi IR komparatif pada Modul 02, 03, dan 04.

**Langkah Selanjutnya:** Lanjutkan ke `02_hqe_and_bm25.ipynb` untuk membangun retriever BM25 dan modul **Historical Query Expansion (HQE)**.